# Taller: App Streamlit para actualizar inventario + chatbot (Parte 3)

Objetivos:
- Construir una App (Streamlit) dentro de Databricks para actualizar en tiempo real la tabla `inventario_insumos_oficina`.
- Agregar un chatbot básico para consultas sobre la tabla.

Referencia: databricks-apps-cookbook
- `https://github.com/databricks-solutions/databricks-apps-cookbook/`

Nota: Esta app puede ejecutarse como Databricks App o desde un notebook (modo demo). Ajusta según tu entorno.



In [ ]:
# Variables de conexión (mismo catálogo/esquema que en la Parte 1)
try:
    dbutils.widgets.text("apellido", "apellido", "Tu apellido")
    APELLIDO = dbutils.widgets.get("apellido").strip().lower()
except Exception:
    APELLIDO = "apellido"

CATALOGO = f"databricks_workshop_{APELLIDO}"
ESQUEMA = "gold"
TABLA = "inventario_insumos_oficina"

spark.sql(f"USE CATALOG `{CATALOGO}`")
spark.sql(f"USE `{CATALOGO}`.`{ESQUEMA}`")
print(f"Usando {CATALOGO}.{ESQUEMA}.{TABLA}")


In [ ]:
# App Streamlit (modo notebook demo). Para Apps, coloca este código en tu App .py
import streamlit as st
from pyspark.sql.functions import col

st.set_page_config(page_title="Inventario Oficina", layout="wide")
st.title("Inventario de Insumos de Oficina")

# Cargar datos
@st.cache_data(ttl=30)
def cargar_inventario():
    return spark.table(f"{CATALOGO}.{ESQUEMA}.{TABLA}")

df = cargar_inventario()
st.write("Vista previa:")
st.dataframe(df.limit(50).toPandas())

st.subheader("Actualizar stock")
item_id = st.text_input("Item ID (ej.: ITM0001)")
nuevo_stock = st.number_input("Nuevo stock", min_value=0, step=1)

if st.button("Actualizar"):
    if item_id:
        spark.sql(f"UPDATE `{CATALOGO}`.`{ESQUEMA}`.`{TABLA}` SET stock_actual = {int(nuevo_stock)} WHERE item_id = '{item_id}'")
        st.success(f"Stock actualizado para {item_id} → {int(nuevo_stock)}")
        st.cache_data.clear()
        df = cargar_inventario()
    else:
        st.warning("Ingresa un Item ID válido.")

st.subheader("Filtrar por categoría / subcategoría")
cat_list = [r[0] for r in spark.sql(f"SELECT DISTINCT categoria FROM `{CATALOGO}`.`{ESQUEMA}`.`{TABLA}`").collect()]
cat_sel = st.selectbox("Categoría", options=["(todas)"] + cat_list)

sub_list = []
if cat_sel and cat_sel != "(todas)":
    sub_list = [r[0] for r in spark.sql(f"SELECT DISTINCT subcategoria FROM `{CATALOGO}`.`{ESQUEMA}`.`{TABLA}` WHERE categoria = '{cat_sel}'").collect()]
sub_sel = st.selectbox("Subcategoría", options=["(todas)"] + sub_list)

query = f"SELECT * FROM `{CATALOGO}`.`{ESQUEMA}`.`{TABLA}`"
conds = []
if cat_sel and cat_sel != "(todas)":
    conds.append(f"categoria = '{cat_sel}'")
if sub_sel and sub_sel != "(todas)":
    conds.append(f"subcategoria = '{sub_sel}'")
if conds:
    query += " WHERE " + " AND ".join(conds)

st.write("Resultados filtrados:")
st.dataframe(spark.sql(query).limit(200).toPandas())


In [ ]:
# Chatbot básico (simula NL→SQL muy simple para demo)
# Para algo más avanzado, integra Genie en el dashboard o un endpoint de IA gobernado.
import re

st.subheader("Chatbot (demo)")
pregunta = st.text_input("Pregunta en español (ej.: 'mostrar insumos por categoría Escritura con stock bajo'):")

sql_generada = None
if st.button("Consultar"):
    q = pregunta.lower()
    if "stock bajo" in q or "por debajo del mínimo" in q:
        sql_generada = f"""
        SELECT item_id, nombre, categoria, subcategoria, stock_actual, stock_minimo
        FROM `{CATALOGO}`.`{ESQUEMA}`.`{TABLA}`
        WHERE stock_actual < stock_minimo
        ORDER BY stock_actual ASC
        LIMIT 50
        """
    elif "por categoría" in q:
        m = re.search(r"categor[ií]a\s+(\w+)", q)
        if m:
            cat_val = m.group(1)
            sql_generada = f"""
            SELECT categoria, subcategoria, COUNT(*) items, SUM(stock_actual) stock_total
            FROM `{CATALOGO}`.`{ESQUEMA}`.`{TABLA}`
            WHERE lower(categoria) = lower('{cat_val}')
            GROUP BY categoria, subcategoria
            ORDER BY stock_total DESC
            LIMIT 50
            """
    else:
        # fallback genérico
        sql_generada = f"SELECT * FROM `{CATALOGO}`.`{ESQUEMA}`.`{TABLA}` LIMIT 50"

    st.code(sql_generada, language="sql")
    st.dataframe(spark.sql(sql_generada).toPandas())
